In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

100%|██████████| 9.91M/9.91M [00:00<00:00, 41.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.09MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.85MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.80MB/s]


In [3]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),   # Regularization
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        return self.model(x)

In [4]:
model = MLP()

In [5]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4   # L2 Regularization
)

In [6]:
best_loss = float('inf')
patience = 3
counter = 0

epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # Early Stopping
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping triggered")
        break

Epoch 1, Train Loss: 337.1033, Val Loss: 23.7549
Epoch 2, Train Loss: 165.0443, Val Loss: 17.2897
Epoch 3, Train Loss: 135.3996, Val Loss: 14.5730
Epoch 4, Train Loss: 115.0974, Val Loss: 13.0063
Epoch 5, Train Loss: 108.4803, Val Loss: 12.3963
Epoch 6, Train Loss: 98.6452, Val Loss: 12.1800
Epoch 7, Train Loss: 91.5116, Val Loss: 11.3504
Epoch 8, Train Loss: 87.4421, Val Loss: 11.2881
Epoch 9, Train Loss: 83.4135, Val Loss: 11.0354
Epoch 10, Train Loss: 81.5877, Val Loss: 11.3173


In [7]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 97.87%


In [8]:
learning_rates = [0.01, 0.001]

for lr in learning_rates:
    print(f"\nTraining with lr = {lr}")

    model = MLP()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(2):
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()


Training with lr = 0.01

Training with lr = 0.001
